# 02 - Analysis and Visualization

This notebook transforms cleaned tables into comparable indicators and core figures.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = BASE_DIR / 'database' / 'database.db'
OUT_DIR = BASE_DIR / 'outputs' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(DB_PATH) as conn:
    norway = pd.read_sql_query('SELECT * FROM norway_inequality_indicators_clean ORDER BY year', conn)
    usa = pd.read_sql_query('SELECT * FROM usa_clean', conn)
    ph = pd.read_sql_query("SELECT * FROM philippines_clean WHERE region = 'Philippines' LIMIT 1", conn)

ph_row = ph.iloc[0]
norway.head()

In [ ]:
def usa_metric(df, metric):
    row = df[(df['income_type'] == 'MONEY INCOME') & (df['measure'] == metric)]
    if row.empty:
        return []
    row = row.iloc[0]
    points = []
    if pd.notna(row['year_2023_estimate']):
        points.append((2023, float(row['year_2023_estimate'])))
    if pd.notna(row['year_2024_estimate']):
        points.append((2024, float(row['year_2024_estimate'])))
    return points

ph_points = [
    (2009, ph_row['gini_2009']),
    (2012, ph_row['gini_2012']),
    (2015, ph_row['gini_2015']),
    (2018, ph_row['gini_2018']),
    (2021, ph_row['gini_2021']),
    (2023, ph_row['gini_2023']),
]

In [ ]:
norway_series = norway[['year', 'gini_all_population']].rename(columns={'gini_all_population': 'gini'}).copy()
norway_series['country'] = 'Norway'

usa_series = pd.DataFrame(usa_metric(usa, 'Gini index of income inequality'), columns=['year', 'gini'])
usa_series['country'] = 'USA'

ph_series = pd.DataFrame(ph_points, columns=['year', 'gini']).dropna()
ph_series['country'] = 'Philippines'

gini_long = pd.concat([norway_series, usa_series, ph_series], ignore_index=True)
gini_long.head()

In [ ]:
pivot = gini_long.pivot_table(index='year', columns='country', values='gini', aggfunc='mean')

ax = pivot.plot(figsize=(10, 5), marker='o', title='Gini trends: Norway, USA, Philippines')
ax.set_ylabel('Gini coefficient')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
latest = gini_long.sort_values('year').groupby('country', as_index=False).tail(1).reset_index(drop=True)
latest = latest.sort_values('gini', ascending=False)

ax = latest.plot(kind='bar', x='country', y='gini', figsize=(7, 4), legend=False, title='Latest available Gini by country')
ax.set_ylabel('Gini coefficient')
ax.set_xlabel('Country')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

latest

In [ ]:
norway_indicators = norway[['year', 'gini_all_population', 'p90_p10_all_population', 's80_s20_all_population']].set_index('year')

ax = norway_indicators.plot(figsize=(10, 5), marker='o', title='Norway: Gini, P90/P10, S80/S20')
ax.set_ylabel('Indicator value')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
comparison = latest.rename(columns={'year': 'latest_year', 'gini': 'latest_gini'}).copy()
comparison['rank_high_to_low_inequality'] = comparison['latest_gini'].rank(method='dense', ascending=False).astype(int)
comparison = comparison.sort_values('rank_high_to_low_inequality')
comparison

## Analysis Summary

- The harmonized Gini trend highlights cross-country level differences.
- Norway has long-run indicator depth that supports multi-metric trend analysis.
- USA and Philippines provide shorter but meaningful benchmark points.

Next notebook focus: synthesize findings into insights, caveats, and policy interpretation.